# 将 Redis 用作 OpenAI 的向量数据库

本 notebook 介绍如何将 Redis 用作存储 OpenAI 嵌入的向量数据库。Redis 是一个可扩展的实时数据库，在使用 [RediSearch Module](https://oss.redislabs.com/redisearch/) 时可以用作向量数据库。RediSearch 模块允许你在 Redis 中对向量建立索引并进行搜索。本 notebook 将向你展示如何使用 RediSearch 模块，对通过 OpenAI API 创建并存储在 Redis 中的向量建立索引和进行搜索。

### 什么是 Redis？

大多数有 Web 服务背景的开发者可能都熟悉 Redis。从本质上讲，Redis 是一个开源的键值存储，可以用作缓存、消息代理和数据库。开发者选择 Redis 是因为它速度快、拥有庞大的客户端库生态，并且多年来已被各大企业部署使用。

除了 Redis 的传统用途之外，Redis 还提供了 [Redis Modules](https://redis.io/modules)，这是一种通过新的数据类型和命令来扩展 Redis 的方式。示例模块包括 [RedisJSON](https://redis.io/docs/stack/json/)、[RedisTimeSeries](https://redis.io/docs/stack/timeseries/)、[RedisBloom](https://redis.io/docs/stack/bloom/) 和 [RediSearch](https://redis.io/docs/stack/search/)。

### 什么是 RediSearch？

RediSearch 是一个 [Redis 模块](https://redis.io/modules)，为 Redis 提供查询、二级索引、全文搜索和向量搜索功能。要使用 RediSearch，首先要在 Redis 数据上声明索引，然后就可以使用 RediSearch 客户端查询这些数据。有关 RediSearch 功能集的更多信息，请参阅 [README](./README.md) 或 [RediSearch 文档](https://redis.io/docs/stack/search/)。

### 部署选项

部署 Redis 的方式有很多。对于本地开发，最快的方法是使用 [Redis Stack docker 容器](https://hub.docker.com/r/redis/redis-stack)，我们在这里也将使用它。Redis Stack 包含多个 Redis 模块，可以组合使用来构建一个快速的多模型数据存储和查询引擎。

对于生产用例，最简单的入门方式是使用 [Redis Cloud](https://redislabs.com/redis-enterprise-cloud/overview/) 服务。Redis Cloud 是一项全托管的 Redis 服务。你也可以使用 [Redis Enterprise](https://redislabs.com/redis-enterprise/overview/) 在自己的基础设施上部署 Redis。Redis Enterprise 是一项全托管的 Redis 服务，可以部署在 kubernetes、本地或云端。

此外，各大云服务商（[AWS Marketplace](https://aws.amazon.com/marketplace/pp/prodview-e6y7ork67pjwg?sr=0-2&ref_=beagle&applicationId=AWSMPContessa)、[Google Marketplace](https://console.cloud.google.com/marketplace/details/redislabs-public/redis-enterprise?pli=1) 或 [Azure Marketplace](https://azuremarketplace.microsoft.com/en-us/marketplace/apps/garantiadata.redis_enterprise_1sp_public_preview?tab=Overview)）都在其应用市场中提供 Redis Enterprise。



## 前提条件

在开始这个项目之前，我们需要完成以下设置：

* 启动一个带有 RediSearch 的 Redis 数据库（redis-stack）
* 安装库
    * [Redis-py](https://github.com/redis/redis-py)
* 获取你的 [OpenAI API 密钥](https://beta.openai.com/account/api-keys)

===========================================================

### 启动 Redis

为了让示例保持简单，我们将使用 Redis Stack docker 容器，可以按如下方式启动

```bash
$ docker-compose up -d
```

其中还包含用于管理 Redis 数据库的 [RedisInsight](https://redis.com/redis-enterprise/redis-insight/) 图形界面，启动 docker 容器后，你可以在 [http://localhost:8001](http://localhost:8001) 查看它。

一切准备就绪！接下来，我们导入并创建客户端，用于与刚刚创建的 Redis 数据库通信。

## 安装依赖

Redis-Py 是用于与 Redis 通信的 Python 客户端。我们将用它与 Redis-stack 数据库通信。 

In [ ]:
! pip install redis wget pandas openai

===========================================================
## 准备 OpenAI API 密钥

`OpenAI API key` 用于对查询数据进行向量化。

如果你还没有 OpenAI API 密钥，可以从 [https://beta.openai.com/account/api-keys](https://beta.openai.com/account/api-keys) 获取。

拿到密钥后，请使用以下命令将其添加到环境变量 `OPENAI_API_KEY` 中：

In [ ]:
! export OPENAI_API_KEY="your API key"

In [2]:
# Test that your OpenAI API key is correctly set as an environment variable
# Note. if you run this notebook locally, you will need to reload your terminal and the notebook for the env variables to be live.
import os
import openai

# Note. alternatively you can set a temporary env variable like this:
# os.environ["OPENAI_API_KEY"] = 'sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

if os.getenv("OPENAI_API_KEY") is not None:
    openai.api_key = os.getenv("OPENAI_API_KEY")
    print ("OPENAI_API_KEY is ready")
else:
    print ("OPENAI_API_KEY environment variable not found")

OPENAI_API_KEY is ready


## 加载数据

在本节中，我们将加载已经转换为向量的嵌入数据。我们将使用这些数据在 Redis 中创建索引，然后搜索相似的向量。

In [3]:
import sys
import numpy as np
import pandas as pd
from typing import List

# use helper function in nbutils.py to download and read the data
# this should take from 5-10 min to run
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())
import nbutils

nbutils.download_wikipedia_data()
data = nbutils.read_wikipedia_data()

data.head()

File Downloaded


,id,url,title,text,title_vector,content_vector,vector_id
0,1,https://simple.wikipedia.org/wiki/April,April,April is the fourth month of the year in the J...,"[0.001009464613161981, -0.020700545981526375, ...","[-0.011253940872848034, -0.013491976074874401,...",0
1,2,https://simple.wikipedia.org/wiki/August,August,August (Aug.) is the eighth month of the year ...,"[0.0009286514250561595, 0.000820168002974242, ...","[0.0003609954728744924, 0.007262262050062418, ...",1
2,6,https://simple.wikipedia.org/wiki/Art,Art,Art is a creative activity that expresses imag...,"[0.003393713850528002, 0.0061537534929811954, ...","[-0.004959689453244209, 0.015772193670272827, ...",2
3,8,https://simple.wikipedia.org/wiki/A,A,A or a is the first letter of the English alph...,"[0.0153952119871974, -0.013759135268628597, 0....","[0.024894846603274345, -0.022186409682035446, ...",3
4,9,https://simple.wikipedia.org/wiki/Air,Air,Air refers to the Earth's atmosphere. Air is a...,"[0.02224554680287838, -0.02044147066771984, -0...","[0.021524671465158463, 0.018522677943110466, -...",4


## 连接 Redis

Redis 数据库已经运行起来了，现在可以使用 Redis-py 客户端连接它。我们将使用 Redis 数据库的默认主机和端口，即 `localhost:6379`。



In [4]:
import redis
from redis.commands.search.indexDefinition import (
    IndexDefinition,
    IndexType
)
from redis.commands.search.query import Query
from redis.commands.search.field import (
    TextField,
    VectorField
)

REDIS_HOST =  "localhost"
REDIS_PORT = 6379
REDIS_PASSWORD = "" # default for passwordless Redis

# Connect to Redis
redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD
)
redis_client.ping()

True

## 在 Redis 中创建搜索索引

下面的单元将展示如何在 Redis 中指定并创建搜索索引。我们将：

1. 设置一些用于定义索引的常量，例如距离度量和索引名称
2. 使用 RediSearch 字段定义索引 schema
3. 创建索引

In [5]:
# Constants
VECTOR_DIM = len(data['title_vector'][0]) # length of the vectors
VECTOR_NUMBER = len(data)                 # initial number of vectors
INDEX_NAME = "embeddings-index"           # name of the search index
PREFIX = "doc"                            # prefix for the document keys
DISTANCE_METRIC = "COSINE"                # distance metric for the vectors (ex. COSINE, IP, L2)

In [6]:
# Define RediSearch fields for each of the columns in the dataset
title = TextField(name="title")
url = TextField(name="url")
text = TextField(name="text")
title_embedding = VectorField("title_vector",
    "FLAT", {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIM,
        "DISTANCE_METRIC": DISTANCE_METRIC,
        "INITIAL_CAP": VECTOR_NUMBER,
    }
)
text_embedding = VectorField("content_vector",
    "FLAT", {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIM,
        "DISTANCE_METRIC": DISTANCE_METRIC,
        "INITIAL_CAP": VECTOR_NUMBER,
    }
)
fields = [title, url, text, title_embedding, text_embedding]

In [7]:
# Check if index exists
try:
    redis_client.ft(INDEX_NAME).info()
    print("Index already exists")
except:
    # Create RediSearch Index
    redis_client.ft(INDEX_NAME).create_index(
        fields = fields,
        definition = IndexDefinition(prefix=[PREFIX], index_type=IndexType.HASH)
)

## 将文档加载到索引中

有了搜索索引之后，就可以向其中加载文档了。我们将使用与前面示例相同的文档。在 Redis 中，可以使用 HASH 或 JSON（如果在 RediSearch 之外还使用了 RedisJSON）数据类型来存储文档。本示例将使用 HASH 数据类型。下面的单元将展示如何将文档加载到索引中。

In [8]:
def index_documents(client: redis.Redis, prefix: str, documents: pd.DataFrame):
    records = documents.to_dict("records")
    for doc in records:
        key = f"{prefix}:{str(doc['id'])}"

        # create byte vectors for title and content
        title_embedding = np.array(doc["title_vector"], dtype=np.float32).tobytes()
        content_embedding = np.array(doc["content_vector"], dtype=np.float32).tobytes()

        # replace list of floats with byte vectors
        doc["title_vector"] = title_embedding
        doc["content_vector"] = content_embedding

        client.hset(key, mapping = doc)

In [9]:
index_documents(redis_client, PREFIX, data)
print(f"Loaded {redis_client.info()['db0']['keys']} documents in Redis search index with name: {INDEX_NAME}")

Loaded 25000 documents in Redis search index with name: embeddings-index


## 使用 OpenAI 查询嵌入进行简单的向量搜索查询

现在我们有了搜索索引，并且已将文档加载进去，就可以运行搜索查询了。下面我们将提供一个函数，用于运行搜索查询并返回结果。使用这个函数，我们会运行几个查询，展示如何将 Redis 用作向量数据库。

In [10]:
def search_redis(
    redis_client: redis.Redis,
    user_query: str,
    index_name: str = "embeddings-index",
    vector_field: str = "title_vector",
    return_fields: list = ["title", "url", "text", "vector_score"],
    hybrid_fields = "*",
    k: int = 20,
    print_results: bool = True,
) -> List[dict]:

    # Creates embedding vector from user query
    embedded_query = openai.Embedding.create(input=user_query,
                                            model="text-embedding-3-small",
                                            )["data"][0]['embedding']

    # Prepare the Query
    base_query = f'{hybrid_fields}=>[KNN {k} @{vector_field} $vector AS vector_score]'
    query = (
        Query(base_query)
         .return_fields(*return_fields)
         .sort_by("vector_score")
         .paging(0, k)
         .dialect(2)
    )
    params_dict = {"vector": np.array(embedded_query).astype(dtype=np.float32).tobytes()}

    # perform vector search
    results = redis_client.ft(index_name).search(query, params_dict)
    if print_results:
        for i, article in enumerate(results.docs):
            score = 1 - float(article.vector_score)
            print(f"{i}. {article.title} (Score: {round(score ,3) })")
    return results.docs

In [11]:
# For using OpenAI to generate query embedding
results = search_redis(redis_client, 'modern art in Europe', k=10)

0. Museum of Modern Art (Score: 0.875)
1. Western Europe (Score: 0.868)
2. Renaissance art (Score: 0.864)
3. Pop art (Score: 0.86)
4. Northern Europe (Score: 0.855)
5. Hellenistic art (Score: 0.853)
6. Modernist literature (Score: 0.847)
7. Art film (Score: 0.843)
8. Central Europe (Score: 0.843)
9. European (Score: 0.841)


In [12]:
results = search_redis(redis_client, 'Famous battles in Scottish history', vector_field='content_vector', k=10)

0. Battle of Bannockburn (Score: 0.869)
1. Wars of Scottish Independence (Score: 0.861)
2. 1651 (Score: 0.853)
3. First War of Scottish Independence (Score: 0.85)
4. Robert I of Scotland (Score: 0.846)
5. 841 (Score: 0.844)
6. 1716 (Score: 0.844)
7. 1314 (Score: 0.837)
8. 1263 (Score: 0.836)
9. William Wallace (Score: 0.835)


## 使用 Redis 进行混合查询

前面的示例展示了如何使用 RediSearch 运行向量搜索查询。在本节中，我们将展示如何将向量搜索与其他 RediSearch 字段结合起来进行混合搜索。在下面的示例中，我们将把向量搜索与全文搜索结合起来。

In [13]:
def create_hybrid_field(field_name: str, value: str) -> str:
    return f'@{field_name}:"{value}"'

# search the content vector for articles about famous battles in Scottish history and only include results with Scottish in the title
results = search_redis(redis_client,
                       "Famous battles in Scottish history",
                       vector_field="title_vector",
                       k=5,
                       hybrid_fields=create_hybrid_field("title", "Scottish")
                       )

0. First War of Scottish Independence (Score: 0.892)
1. Wars of Scottish Independence (Score: 0.889)
2. Second War of Scottish Independence (Score: 0.879)
3. List of Scottish monarchs (Score: 0.873)
4. Scottish Borders (Score: 0.863)


In [14]:
# run a hybrid query for articles about Art in the title vector and only include results with the phrase "Leonardo da Vinci" in the text
results = search_redis(redis_client,
                       "Art",
                       vector_field="title_vector",
                       k=5,
                       hybrid_fields=create_hybrid_field("text", "Leonardo da Vinci")
                       )

# find specific mention of Leonardo da Vinci in the text that our full-text-search query returned
mention = [sentence for sentence in results[0].text.split("\n") if "Leonardo da Vinci" in sentence][0]
mention

0. Art (Score: 1.0)
1. Paint (Score: 0.896)
2. Renaissance art (Score: 0.88)
3. Painting (Score: 0.874)
4. Renaissance (Score: 0.846)


'In Europe, after the Middle Ages, there was a "Renaissance" which means "rebirth". People rediscovered science and artists were allowed to paint subjects other than religious subjects. People like Michelangelo and Leonardo da Vinci still painted religious pictures, but they also now could paint mythological pictures too. These artists also invented perspective where things in the distance look smaller in the picture. This was new because in the Middle Ages people would paint all the figures close up and just overlapping each other. These artists used nudity regularly in their art.'

## HNSW 索引

到目前为止，我们一直使用 ``FLAT``（即"暴力"）索引来运行查询。Redis 还支持 ``HNSW`` 索引，这是一种快速的近似索引。``HNSW`` 索引是一种基于图的索引，它使用分层可导航小世界（hierarchical navigable small world）图来存储向量。对于希望运行近似查询的大型数据集，``HNSW`` 索引是一个不错的选择。

在大多数情况下，``HNSW`` 比 ``FLAT`` 需要更长的构建时间、占用更多内存，但查询运行速度更快，尤其是在大型数据集上。

下面的单元将展示如何使用与之前相同的数据创建 ``HNSW`` 索引并用它运行查询。

In [15]:
# re-define RediSearch vector fields to use HNSW index
title_embedding = VectorField("title_vector",
    "HNSW", {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIM,
        "DISTANCE_METRIC": DISTANCE_METRIC,
        "INITIAL_CAP": VECTOR_NUMBER
    }
)
text_embedding = VectorField("content_vector",
    "HNSW", {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIM,
        "DISTANCE_METRIC": DISTANCE_METRIC,
        "INITIAL_CAP": VECTOR_NUMBER
    }
)
fields = [title, url, text, title_embedding, text_embedding]

In [16]:
import time
# Check if index exists
HNSW_INDEX_NAME = INDEX_NAME+ "_HNSW"

try:
    redis_client.ft(HNSW_INDEX_NAME).info()
    print("Index already exists")
except:
    # Create RediSearch Index
    redis_client.ft(HNSW_INDEX_NAME).create_index(
        fields = fields,
        definition = IndexDefinition(prefix=[PREFIX], index_type=IndexType.HASH)
    )

# since RediSearch creates the index in the background for existing documents, we will wait until
# indexing is complete before running our queries. Although this is not necessary for the first query,
# some queries may take longer to run if the index is not fully built. In general, Redis will perform
# best when adding new documents to existing indices rather than new indices on existing documents.
while redis_client.ft(HNSW_INDEX_NAME).info()["indexing"] == "1":
    time.sleep(5)

In [17]:
results = search_redis(redis_client, 'modern art in Europe', index_name=HNSW_INDEX_NAME, k=10)

0. Western Europe (Score: 0.868)
1. Northern Europe (Score: 0.855)
2. Central Europe (Score: 0.843)
3. European (Score: 0.841)
4. Eastern Europe (Score: 0.839)
5. Europe (Score: 0.839)
6. Western European Union (Score: 0.837)
7. Southern Europe (Score: 0.831)
8. Western civilization (Score: 0.83)
9. Council of Europe (Score: 0.827)


In [18]:
# compare the results of the HNSW index to the FLAT index and time both queries
def time_queries(iterations: int = 10):
    print(" ----- Flat Index ----- ")
    t0 = time.time()
    for i in range(iterations):
        results_flat = search_redis(redis_client, 'modern art in Europe', k=10, print_results=False)
    t0 = (time.time() - t0) / iterations
    results_flat = search_redis(redis_client, 'modern art in Europe', k=10, print_results=True)
    print(f"Flat index query time: {round(t0, 3)} seconds\n")
    time.sleep(1)
    print(" ----- HNSW Index ------ ")
    t1 = time.time()
    for i in range(iterations):
        results_hnsw = search_redis(redis_client, 'modern art in Europe', index_name=HNSW_INDEX_NAME, k=10, print_results=False)
    t1 = (time.time() - t1) / iterations
    results_hnsw = search_redis(redis_client, 'modern art in Europe', index_name=HNSW_INDEX_NAME, k=10, print_results=True)
    print(f"HNSW index query time: {round(t1, 3)} seconds")
    print(" ------------------------ ")
time_queries()

 ----- Flat Index ----- 
0. Museum of Modern Art (Score: 0.875)
1. Western Europe (Score: 0.867)
2. Renaissance art (Score: 0.864)
3. Pop art (Score: 0.861)
4. Northern Europe (Score: 0.855)
5. Hellenistic art (Score: 0.853)
6. Modernist literature (Score: 0.847)
7. Art film (Score: 0.843)
8. Central Europe (Score: 0.843)
9. Art (Score: 0.842)
Flat index query time: 0.263 seconds

 ----- HNSW Index ------ 
0. Western Europe (Score: 0.867)
1. Northern Europe (Score: 0.855)
2. Central Europe (Score: 0.843)
3. European (Score: 0.841)
4. Eastern Europe (Score: 0.839)
5. Europe (Score: 0.839)
6. Western European Union (Score: 0.837)
7. Southern Europe (Score: 0.831)
8. Western civilization (Score: 0.83)
9. Council of Europe (Score: 0.827)
HNSW index query time: 0.129 seconds
 ------------------------ 
